# Independent Historical VaR Backtest Validation

Notebook này kiểm tra độc lập việc đánh dấu Historical VaR violation.

Phạm vi kiểm tra:

- independent sign audit;
- first, middle and last forecast audit;
- full violation-table audit;
- chart sign audit.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKTEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_var_backtest.csv"
)

VIOLATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_var_violations.csv"
)

backtest = pd.read_csv(
    BACKTEST_PATH,
    parse_dates=[
        "window_start_date",
        "window_end_date",
        "forecast_date",
        "target_date",
    ],
)

violations = pd.read_csv(
    VIOLATION_PATH,
    parse_dates=["target_date"],
)

print("Backtest shape:", backtest.shape)
print("Violation shape:", violations.shape)
print(backtest.columns.tolist())

Backtest shape: (1387, 9)
Violation shape: (75, 4)
['window_start_date', 'window_end_date', 'forecast_date', 'target_date', 'observations', 'quantile_return', 'historical_var', 'target_return', 'violation']


Independent sign audit

chọn bốn trường hợp:
- normal positive return;
- normal negative return;
- obvious violation;
- near-threshold observation.

In [3]:
normal_positive = (
    backtest.loc[
        (backtest["target_return"] > 0)
        & (~backtest["violation"])
    ]
    .iloc[0]
)

normal_negative = (
    backtest.loc[
        (backtest["target_return"] < 0)
        & (~backtest["violation"])
    ]
    .iloc[0]
)

violation_rows = (
    backtest.loc[
        backtest["violation"]
    ]
    .copy()
)

violation_rows["breach_depth"] = (
    violation_rows["quantile_return"]
    - violation_rows["target_return"]
)

obvious_violation = (
    violation_rows
    .sort_values(
        "breach_depth",
        ascending=False,
    )
    .iloc[0]
)

near_threshold_data = backtest.copy()

near_threshold_data["threshold_gap"] = (
    near_threshold_data["target_return"]
    - near_threshold_data["quantile_return"]
)

near_threshold = (
    near_threshold_data.iloc[
        np.abs(
            near_threshold_data[
                "threshold_gap"
            ].to_numpy()
        ).argmin()
    ]
)

So sánh Expected và Actual

In [4]:
audit_cases = pd.DataFrame(
    [
        {
            "case": "normal_positive",
            "target_date": normal_positive["target_date"],
            "target_return": normal_positive["target_return"],
            "quantile_return": normal_positive["quantile_return"],
            "expected_violation": (
                normal_positive["target_return"]
                < normal_positive["quantile_return"]
            ),
            "actual_violation": normal_positive["violation"],
        },
        {
            "case": "normal_negative",
            "target_date": normal_negative["target_date"],
            "target_return": normal_negative["target_return"],
            "quantile_return": normal_negative["quantile_return"],
            "expected_violation": (
                normal_negative["target_return"]
                < normal_negative["quantile_return"]
            ),
            "actual_violation": normal_negative["violation"],
        },
        {
            "case": "obvious_violation",
            "target_date": obvious_violation["target_date"],
            "target_return": obvious_violation["target_return"],
            "quantile_return": obvious_violation["quantile_return"],
            "expected_violation": (
                obvious_violation["target_return"]
                < obvious_violation["quantile_return"]
            ),
            "actual_violation": obvious_violation["violation"],
        },
        {
            "case": "near_threshold",
            "target_date": near_threshold["target_date"],
            "target_return": near_threshold["target_return"],
            "quantile_return": near_threshold["quantile_return"],
            "expected_violation": (
                near_threshold["target_return"]
                < near_threshold["quantile_return"]
            ),
            "actual_violation": near_threshold["violation"],
        },
    ]
)

audit_cases["matched"] = (
    audit_cases["expected_violation"]
    == audit_cases["actual_violation"]
)

audit_cases

,case,target_date,target_return,quantile_return,expected_violation,actual_violation,matched
0,normal_positive,2020-12-31,0.014634,-0.037728,False,False,True
1,normal_negative,2021-01-07,-0.000895,-0.037728,False,False,True
2,obvious_violation,2025-04-03,-0.069840,-0.017793,True,True,True
3,near_threshold,2021-06-08,-0.024962,-0.024782,True,True,True


In [5]:
assert audit_cases["matched"].all()

print("B-06.1 Independent sign audit: PASS")

B-06.1 Independent sign audit: PASS


First, middle, last audit
 chọn:

- forecast đầu tiên;
- forecast ở giữa;
- forecast cuối cùng.

Với mỗi forecast, kiểm tra:

- target return;
- quantile threshold;
- expected violation;
- actual violation.

In [6]:
positions = {
    "first": 0,
    "middle": len(backtest) // 2,
    "last": len(backtest) - 1,
}

position_audit = []

for label, position in positions.items():
    row = backtest.iloc[position]

    expected_violation = (
        row["target_return"]
        < row["quantile_return"]
    )

    position_audit.append(
        {
            "forecast": label,
            "position": position,
            "forecast_date": row["forecast_date"],
            "target_date": row["target_date"],
            "target_return": row["target_return"],
            "quantile_return": row["quantile_return"],
            "expected_violation": expected_violation,
            "actual_violation": row["violation"],
            "matched": (
                expected_violation
                == row["violation"]
            ),
        }
    )

position_audit = pd.DataFrame(
    position_audit
)

position_audit

,forecast,position,forecast_date,target_date,target_return,quantile_return,expected_violation,actual_violation,matched
0,first,0,2020-12-30,2020-12-31,0.014634,-0.037728,False,False,True
1,middle,693,2023-10-12,2023-10-13,0.005953,-0.030237,False,False,True
2,last,1386,2026-07-27,2026-07-28,0.024935,-0.025663,False,False,True


In [7]:
assert position_audit["matched"].all()

print(
    "B-06.2 First/Middle/Last audit: PASS"
)

B-06.2 First/Middle/Last audit: PASS


Audit toàn bộ violation table

In [8]:
expected_violation = (
    backtest["target_return"]
    < backtest["quantile_return"]
)

flag_mismatches = (
    backtest["violation"]
    != expected_violation
)

violation_rows_valid = (
    backtest.loc[
        backtest["violation"],
        "target_return",
    ]
    <
    backtest.loc[
        backtest["violation"],
        "quantile_return",
    ]
).all()

non_violation_rows_valid = (
    backtest.loc[
        ~backtest["violation"],
        "target_return",
    ]
    >=
    backtest.loc[
        ~backtest["violation"],
        "quantile_return",
    ]
).all()

print(
    "Rows:",
    len(backtest),
)

print(
    "Violations:",
    int(backtest["violation"].sum()),
)

print(
    "Non-violations:",
    int((~backtest["violation"]).sum()),
)

print(
    "Flag mismatches:",
    int(flag_mismatches.sum()),
)

print(
    "Violation rows valid:",
    violation_rows_valid,
)

print(
    "Non-violation rows valid:",
    non_violation_rows_valid,
)

Rows: 1387
Violations: 75
Non-violations: 1312
Flag mismatches: 0
Violation rows valid: True
Non-violation rows valid: True


Kiểm tra violation file

In [9]:
violation_file_valid = (
    violations["target_return"]
    < violations["quantile_return"]
).all()

backtest_violation_subset = (
    backtest.loc[
        backtest["violation"],
        [
            "target_date",
            "target_return",
            "quantile_return",
            "historical_var",
        ],
    ]
    .reset_index(drop=True)
)

same_violation_count = (
    len(backtest_violation_subset)
    == len(violations)
)

numeric_columns = [
    "target_return",
    "quantile_return",
    "historical_var",
]

same_numeric_values = np.allclose(
    backtest_violation_subset[
        numeric_columns
    ].to_numpy(),
    violations[
        numeric_columns
    ].to_numpy(),
    rtol=0.0,
    atol=1e-12,
)

same_target_dates = (
    backtest_violation_subset[
        "target_date"
    ]
    .equals(
        violations[
            "target_date"
        ]
    )
)

print(
    "Violation CSV rows valid:",
    violation_file_valid,
)

print(
    "Same violation count:",
    same_violation_count,
)

print(
    "Same target dates:",
    same_target_dates,
)

print(
    "Same numeric values:",
    same_numeric_values,
)

Violation CSV rows valid: True
Same violation count: True
Same target dates: True
Same numeric values: True


Assertions

In [10]:
assert len(backtest) == 1387
assert int(backtest["violation"].sum()) == 75
assert int(flag_mismatches.sum()) == 0
assert violation_rows_valid
assert non_violation_rows_valid
assert violation_file_valid
assert same_violation_count
assert same_target_dates
assert same_numeric_values

print(
    "B-06.3 Full violation-table audit: PASS"
)

B-06.3 Full violation-table audit: PASS


Audit chart

kiểm tra quy ước dấu trong toàn bộ dữ liệu

In [11]:
chart_audit = pd.DataFrame(
    {
        "check": [
            "quantile_return is non-positive",
            "historical_var is non-negative",
            "historical_var equals max(0, -quantile_return)",
            "violation points lie below quantile threshold",
        ],
        "passed": [
            (backtest["quantile_return"] <= 0.0).all(),
            (backtest["historical_var"] >= 0.0).all(),
            np.allclose(
                backtest["historical_var"].to_numpy(),
                np.maximum(
                    0.0,
                    -backtest["quantile_return"].to_numpy(),
                ),
                rtol=0.0,
                atol=1e-12,
            ),
            (
                backtest.loc[
                    backtest["violation"],
                    "target_return",
                ]
                <
                backtest.loc[
                    backtest["violation"],
                    "quantile_return",
                ]
            ).all(),
        ],
    }
)

chart_audit

,check,passed
0,quantile_return is non-positive,True
1,historical_var is non-negative,True
2,"historical_var equals max(0, -quantile_return)",True
3,violation points lie below quantile threshold,True


In [12]:
assert chart_audit["passed"].all()

print("B-06.4 Chart sign audit: PASS")

B-06.4 Chart sign audit: PASS


xác nhận cột nào phải dùng để vẽ

In [13]:
chart_columns = {
    "actual_return_series": "target_return",
    "negative_threshold_series": "quantile_return",
    "positive_var_magnitude": "historical_var",
}

assert chart_columns["actual_return_series"] in backtest.columns
assert chart_columns["negative_threshold_series"] in backtest.columns
assert chart_columns["positive_var_magnitude"] in backtest.columns

assert (
    backtest["quantile_return"] <= 0.0
).all()

assert (
    backtest["historical_var"] >= 0.0
).all()

print("Series plotted as actual return: target_return")
print("Series plotted as negative threshold: quantile_return")
print("Series not used as return threshold: historical_var")

Series plotted as actual return: target_return
Series plotted as negative threshold: quantile_return
Series not used as return threshold: historical_var


kiểm tra file hình tồn tại

In [14]:
CHART_PATH = (
    PROJECT_ROOT
    / "figures"
    / "var"
    / "09_historical_var_backtest.png"
)

assert CHART_PATH.exists()
assert CHART_PATH.stat().st_size > 0

print("Chart file:", CHART_PATH)
print("Chart file size:", CHART_PATH.stat().st_size)
print("B-06.4 Chart file audit: PASS")

Chart file: /Users/admin/Documents/project/portfolio-var-risk-system/figures/var/09_historical_var_backtest.png
Chart file size: 795483
B-06.4 Chart file audit: PASS
